# Training Script and Evaluation

first setup 70:30 train, test split 

In [1]:
from typing import List
from pathlib import Path, PosixPath
import glob
import os
import pickle

from termcolor import cprint

from detectron2.data import DatasetCatalog, MetadataCatalog
from detectron2.utils.visualizer import Visualizer, ColorMode

/home/abhinavchadaga/CS/fri_II/.venv/lib/python3.8/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
path_to_dataset = "./dataset/annotated_building_imgs/"

dirs = glob.glob(os.path.join(path_to_dataset, "*"))
num_buildings = len(dirs)
trainset_buildings = []
testset_buildings = []
trainset_size = int(0.7 * num_buildings)
testset_size = num_buildings - trainset_size

for i, b in enumerate(dirs):
    if i < trainset_size:
        trainset_buildings.append(b)
    else:
        testset_buildings.append(b)

cprint(f"trainset size: {trainset_size}", "green", attrs=["bold"])
cprint(f"testset size: {testset_size}\n", "green", attrs=["bold"])

print(f"trainset: ")
for b in trainset_buildings:
    print(b)
print()

print(f"testset: ")
for b in testset_buildings:
    print(b)


trainset size: 7
testset size: 3

trainset: 
./dataset/annotated_building_imgs/the_nine
./dataset/annotated_building_imgs/bergstrom
./dataset/annotated_building_imgs/signature
./dataset/annotated_building_imgs/inspire_on_22nd
./dataset/annotated_building_imgs/26_west
./dataset/annotated_building_imgs/jester_west
./dataset/annotated_building_imgs/cpe

testset: 
./dataset/annotated_building_imgs/jester_east
./dataset/annotated_building_imgs/ruckus_on_rio
./dataset/annotated_building_imgs/ahg


In [8]:
def register_dataset(buildings: List[str]):
    """ Define function to register 
        our custom dataset using the specified buildings
    """
    dataset = list()
    for building in buildings:
        for img_path in os.listdir(building):
            with open(os.path.join(building, img_path), "rb") as img:
                dataset.append(pickle.load(img))

    return dataset

### Register Training Set

In [ ]:
DatasetCatalog.register("train", lambda: register_dataset(trainset_buildings))
MetadataCatalog.get("train").thing_classes = ["label", "button", "not button"]
train_set: List[dict] = DatasetCatalog.get("train")

### Register Test Set

In [ ]:
DatasetCatalog.register("test", lambda: register_dataset(testset_buildings))
MetadataCatalog.get("test").thing_class = ["lablel", "button", "not button"]
test_set: List[dict] = DatasetCatalog.get("test")

### Update Detectron Config to Handle Our Dataset

In [10]:
from detectron2.config import get_cfg

cfg = get_cfg()
cfg.DATASETS.TRAIN = train_set
cfg.DATASETS.TEST = test_set

CfgNode({'VERSION': 2, 'MODEL': CfgNode({'LOAD_PROPOSALS': False, 'MASK_ON': False, 'KEYPOINT_ON': False, 'DEVICE': 'cuda', 'META_ARCHITECTURE': 'GeneralizedRCNN', 'WEIGHTS': '', 'PIXEL_MEAN': [103.53, 116.28, 123.675], 'PIXEL_STD': [1.0, 1.0, 1.0], 'BACKBONE': CfgNode({'NAME': 'build_resnet_backbone', 'FREEZE_AT': 2}), 'FPN': CfgNode({'IN_FEATURES': [], 'OUT_CHANNELS': 256, 'NORM': '', 'FUSE_TYPE': 'sum'}), 'PROPOSAL_GENERATOR': CfgNode({'NAME': 'RPN', 'MIN_SIZE': 0}), 'ANCHOR_GENERATOR': CfgNode({'NAME': 'DefaultAnchorGenerator', 'SIZES': [[32, 64, 128, 256, 512]], 'ASPECT_RATIOS': [[0.5, 1.0, 2.0]], 'ANGLES': [[-90, 0, 90]], 'OFFSET': 0.0}), 'RPN': CfgNode({'HEAD_NAME': 'StandardRPNHead', 'IN_FEATURES': ['res4'], 'BOUNDARY_THRESH': -1, 'IOU_THRESHOLDS': [0.3, 0.7], 'IOU_LABELS': [0, -1, 1], 'BATCH_SIZE_PER_IMAGE': 256, 'POSITIVE_FRACTION': 0.5, 'BBOX_REG_LOSS_TYPE': 'smooth_l1', 'BBOX_REG_LOSS_WEIGHT': 1.0, 'BBOX_REG_WEIGHTS': (1.0, 1.0, 1.0, 1.0), 'SMOOTH_L1_BETA': 0.0, 'LOSS_WEIGH